# P1-E16.7E renamed(P1-E11.E) Resume-Safe Frozen External Re-evaluation

Explanation
The scientific contract is unchanged: AHEPA supports Normal vs AD; Dryad supports all four endpoints; conventional and endpoint-selected deep models remain frozen; no external tuning or recalibration is permitted.


## 1. Configuration, endpoint contract, and chronology guardrails

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, random, sys, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, log_loss, matthews_corrcoef, roc_auc_score, roc_curve)
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as exc:
    raise ImportError('PyTorch is required in the eeg_snn environment.') from exc

SEED=20261010
BOOTSTRAP_REPS=5000
MAX_INPUT_EPOCHS=12
TASKS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],
       'normal_vs_mci':['Normal','MCI'],'three_class':['Normal','MCI','AD']}
TASK_LABELS={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD',
             'normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
EXTERNAL_PROTOCOL={'AHEPA':['normal_vs_ad'],
                   'Dryad':['normal_vs_ad','mci_vs_ad','normal_vs_mci','three_class']}
METHODS=['Frozen conventional benchmark','Endpoint-selected CEEDNet/SSL']
CHANCE_BA={task:1/len(classes) for task,classes in TASKS.items()}

PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
LEGACY_OUTPUTS_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E16_7A_ROOT=OUTPUTS_ROOT/'p1_e16_7a_four_endpoint_nested_conventional_freeze'
E16_7D_ROOT=OUTPUTS_ROOT/'p1_e16_7d_full_development_refit_retrospective_locked_test'
AHEPA_ROOT=LEGACY_OUTPUTS_ROOT/'p1_e10_2_ahepa_frozen_normal_ad_external_validation'
DRYAD_ROOT=LEGACY_OUTPUTS_ROOT/'p1_e10_3_dryad_frozen_three_endpoint_external_validation'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7e_frozen_external_reevaluation_transportability_shift_chronology'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; CHECKPOINTS=OUTPUT_ROOT/'checkpoints'
for folder in (OUTPUT_ROOT,TABLES,FIGURES,CHECKPOINTS): folder.mkdir(parents=True,exist_ok=True)

CONV_MODELS=E16_7A_ROOT/'tables'/'10_FROZEN_ENDPOINT_MODELS.csv'
CONV_INTEGRITY=E16_7A_ROOT/'tables'/'15_FINAL_INTEGRITY_CHECKS.csv'
CONV_RUN=E16_7A_ROOT/'run_record.json'
DEEP_MODELS=E16_7D_ROOT/'tables'/'05_SERIALIZED_MODEL_ARTIFACTS.csv'
DEEP_INTEGRITY=E16_7D_ROOT/'tables'/'09_FINAL_INTEGRITY_CHECKS.csv'
DEEP_RUN=E16_7D_ROOT/'run_record.json'
DATASETS={
 'AHEPA':{'matrix':AHEPA_ROOT/'AHEPA_LOCKED_NONSPIKING_FEATURE_MATRIX.npz',
          'audit':AHEPA_ROOT/'tables'/'05_AHEPA_PARTICIPANT_PREPROCESSING_AUDIT.csv',
          'labels':AHEPA_ROOT/'tables'/'03_SEALED_AHEPA_LABEL_LOCK.csv',
          'prior_run':AHEPA_ROOT/'run_record.json'},
 'Dryad':{'matrix':DRYAD_ROOT/'DRYAD_LOCKED_NONSPIKING_FEATURE_MATRIX.npz',
          'audit':DRYAD_ROOT/'tables'/'04_DRYAD_PARTICIPANT_PREPROCESSING_AUDIT.csv',
          'labels':DRYAD_ROOT/'tables'/'03_SEALED_DRYAD_LABEL_LOCK.csv',
          'prior_run':DRYAD_ROOT/'run_record.json'}}

def sha256_file(path,block=1<<20):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(block),b''): h.update(chunk)
    return h.hexdigest()
def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path
def save_figure(fig,name):
    fig.tight_layout(); fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight')
    fig.savefig(FIGURES/f'{name}.png',dpi=500,bbox_inches='tight',facecolor='white')
def json_default(value):
    if isinstance(value,(np.integer,)): return int(value)
    if isinstance(value,(np.floating,)): return float(value)
    if isinstance(value,np.ndarray): return value.tolist()
    if isinstance(value,Path): return str(value)
    raise TypeError(type(value).__name__)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style='whitegrid',context='paper',font_scale=1.05)
plt.rcParams.update({'figure.dpi':130,'savefig.dpi':500,'pdf.fonttype':42,'ps.fonttype':42})
print({'device':str(DEVICE),'output':str(OUTPUT_ROOT),'evidence':'locked external re-evaluation'})


## 2. Authenticate frozen models and previously locked external hand-offs

In [ ]:
required=[CONV_MODELS,CONV_INTEGRITY,CONV_RUN,DEEP_MODELS,DEEP_INTEGRITY,DEEP_RUN]
for spec in DATASETS.values(): required.extend(spec.values())
missing=[str(path) for path in required if not Path(path).is_file()]
if missing: raise FileNotFoundError('Missing required frozen artifact(s):\n'+'\n'.join(missing))

def passed_table(path):
    q=pd.read_csv(path)
    return 'pass' in q and q['pass'].astype(str).str.lower().eq('true').all()
if not passed_table(CONV_INTEGRITY): raise AssertionError('P1-E16.7A integrity failure.')
if not passed_table(DEEP_INTEGRITY): raise AssertionError('P1-E16.7D integrity failure.')
conv_run=json.loads(CONV_RUN.read_text(encoding='utf-8'))
deep_run=json.loads(DEEP_RUN.read_text(encoding='utf-8'))
if 'PASSED' not in str(conv_run.get('status','')) or deep_run.get('status')!='PASSED':
    raise AssertionError('A frozen upstream model run is not PASSED.')

conv_models=pd.read_csv(CONV_MODELS).rename(columns={'endpoint':'task'})
deep_models=pd.read_csv(DEEP_MODELS)
if set(conv_models.task)!=set(TASKS) or set(deep_models.task)!=set(TASKS):
    raise AssertionError('Frozen model tables do not contain exactly the four endpoints.')
model_checks=[]
for method,table in [('conventional',conv_models),('selected_deep',deep_models)]:
    for row in table.itertuples(index=False):
        path=Path(row.model_path); valid=path.is_file() and sha256_file(path)==row.model_sha256
        model_checks.append({'method':method,'task':row.task,'path':str(path),'expected_sha256':row.model_sha256,'authenticated':valid})
model_checks=pd.DataFrame(model_checks)
if not model_checks.authenticated.all(): display(model_checks); raise AssertionError('Frozen model authentication failed.')

chronology=[]
for dataset,spec in DATASETS.items():
    prior=json.loads(Path(spec['prior_run']).read_text(encoding='utf-8'))
    chronology.append({'dataset':dataset,'prior_external_run':str(spec['prior_run']),
        'prior_run_sha256':sha256_file(spec['prior_run']),'prior_status':prior.get('status',''),
        'historically_untouched_for_P1_E16_7E':False,
        'permitted_claim':'locked external re-evaluation under the final P1-E16.7A/P1-E16.7D model contract'})
chronology=pd.DataFrame(chronology)
inputs=pd.DataFrame([{'role':Path(p).name,'path':str(p),'sha256':sha256_file(p)} for p in required])
save_table(inputs,'01_AUTHENTICATED_INPUTS'); save_table(model_checks,'02_FROZEN_MODEL_AUTHENTICATION')
save_table(chronology,'03_EXTERNAL_CHRONOLOGY_AND_CLAIM_BOUNDARY')
display(model_checks); display(chronology)


## 3. Load external features and epoch arrays while diagnosis labels remain sealed

In [ ]:
external={}; compatibility=[]
for dataset,spec in DATASETS.items():
    with np.load(spec['matrix'],allow_pickle=False) as payload:
        required_keys={'X','participant_id','feature_names'}
        if not required_keys.issubset(payload.files): raise KeyError(f'{dataset} matrix lacks {required_keys-set(payload.files)}')
        X=np.asarray(payload['X'],np.float32); ids=payload['participant_id'].astype(str); names=payload['feature_names'].astype(str)
    audit=pd.read_csv(spec['audit'],dtype={'participant_id':str})
    needed={'participant_id','processing_success','array_path','array_sha256'}
    if not needed.issubset(audit): raise KeyError(f'{dataset} audit lacks {needed-set(audit)}')
    audit=audit.loc[audit.processing_success.astype(str).str.lower().isin(['true','1'])].copy()
    audit=audit[['participant_id','array_path','array_sha256']].drop_duplicates('participant_id')
    matrix=pd.DataFrame({'participant_id':ids,'feature_row':np.arange(len(ids))})
    blind=matrix.merge(audit,on='participant_id',how='inner',validate='one_to_one')
    hashes=blind.apply(lambda r:Path(r.array_path).is_file() and sha256_file(r.array_path)==r.array_sha256,axis=1)
    checks={
      'matrix_shape_valid':X.ndim==2 and X.shape==(len(ids),len(names)),
      'exact_646_feature_schema':X.ndim==2 and X.shape[1]==646,
      'features_finite':np.isfinite(X).all(),
      'participant_ids_unique':len(np.unique(ids))==len(ids),
      'matrix_audit_ids_reconcile':len(blind)==len(ids),
      'epoch_arrays_authenticated':bool(hashes.all())}
    for check,value in checks.items(): compatibility.append({'dataset':dataset,'check':check,'pass':bool(value),'detail':str(X.shape if 'matrix' in check or 'feature' in check else len(ids))})
    external[dataset]={'X':X,'ids':ids,'feature_names':names,'blind':blind}
compatibility=pd.DataFrame(compatibility)
if not compatibility['pass'].all(): display(compatibility); raise AssertionError('External hand-off compatibility failure.')
save_table(compatibility,'04_EXTERNAL_INPUT_COMPATIBILITY'); display(compatibility)
print('External inputs authenticated. Diagnosis locks have not been opened.')


## 4. Reconstruct the selected deep architectures and frozen inference rules

In [ ]:
def load_epochs(path,max_epochs=MAX_INPUT_EPOCHS):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    with np.load(path,allow_pickle=False) as z:
        key=next((k for k in ('epochs_uv','epochs','data','X') if k in z.files),None)
        if key is None: raise KeyError(f'No epoch array in {path}')
        x=np.asarray(z[key],np.float32)
    if x.ndim!=3: raise ValueError(f'Expected [epochs,channels,time], got {x.shape}')
    if x.shape[1]!=19 and x.shape[2]==19: x=x.transpose(0,2,1)
    if x.shape[1]!=19 or not np.isfinite(x).all(): raise ValueError(f'Invalid epoch tensor: {path}, {x.shape}')
    if len(x)>int(max_epochs): x=x[np.linspace(0,len(x)-1,int(max_epochs)).round().astype(int)]
    return x

class AttentionPool(nn.Module):
    def __init__(self,d): super().__init__(); self.score=nn.Linear(d,1)
    def forward(self,z,mask):
        a=self.score(z).squeeze(-1).masked_fill(~mask,-1e9); a=torch.softmax(a,1)
        return (z*a.unsqueeze(-1)).sum(1)
class CEEDNet(nn.Module):
    def __init__(self,n_classes,dropout=.25):
        super().__init__(); self.temporal=nn.Sequential(
            nn.Conv1d(19,38,31,padding=15,groups=19),nn.Conv1d(38,64,1),nn.BatchNorm1d(64),nn.ELU(),
            nn.AvgPool1d(4),nn.Conv1d(64,96,15,padding=7,groups=32),nn.Conv1d(96,96,1),
            nn.BatchNorm1d(96),nn.ELU(),nn.AdaptiveAvgPool1d(1))
        self.pool=AttentionPool(96); self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(96,n_classes))
    def forward(self,x,mask):
        b,e,c,t=x.shape; z=self.temporal(x.reshape(b*e,c,t)).squeeze(-1).reshape(b,e,-1)
        return self.head(self.pool(z,mask))
class MultiResolutionEncoder(nn.Module):
    def __init__(self,d=96):
        super().__init__(); self.branches=nn.ModuleList([nn.Sequential(
            nn.Conv1d(19,32,k,padding=k//2),nn.BatchNorm1d(32),nn.ELU(),nn.AdaptiveAvgPool1d(1)) for k in (7,31,101)])
        self.proj=nn.Linear(96,d)
    def forward(self,x): return F.normalize(self.proj(torch.cat([b(x).squeeze(-1) for b in self.branches],1)),dim=1)
class SSLAttention(nn.Module):
    def __init__(self,n_classes,dropout=.25):
        super().__init__(); self.encoder=MultiResolutionEncoder(96); self.pool=AttentionPool(96)
        self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(96,n_classes))
    def forward(self,x,mask):
        b,e,c,t=x.shape; z=self.encoder(x.reshape(b*e,c,t)).reshape(b,e,-1)
        return self.head(self.pool(z,mask))

def load_deep_state(path):
    try: return torch.load(path,map_location='cpu',weights_only=False)
    except TypeError: return torch.load(path,map_location='cpu')
def build_deep_model(state):
    candidate=str(state.get('candidate','')); dropout=.50 if candidate=='dropout_0.50' else .25
    method=str(state['method'])
    if method=='CEEDNet adaptation': model=CEEDNet(len(state['classes']),dropout)
    elif method=='SSL multiresolution attention': model=SSLAttention(len(state['classes']),dropout)
    else: raise ValueError(f'Unsupported selected method: {method}')
    model.load_state_dict(state['state_dict']); return model.to(DEVICE).eval()
def deep_probability(state,array_path):
    x=load_epochs(array_path,int(state.get('max_input_epochs',MAX_INPUT_EPOCHS)))
    mean=np.asarray(state['channel_mean'],np.float32)[None,:,None]
    sd=np.asarray(state['channel_sd'],np.float32)[None,:,None]
    x=np.clip((x-mean)/sd,-8,8).astype(np.float32)
    xt=torch.from_numpy(x)[None].to(DEVICE); mask=torch.ones((1,len(x)),dtype=torch.bool,device=DEVICE)
    model=build_deep_model(state)
    with torch.no_grad(): probability=torch.softmax(model(xt,mask),1)[0].cpu().numpy()
    del model
    return probability
def conventional_probability(bundle,X,external_names):
    indices=np.asarray(bundle['selected_feature_indices'],int)
    expected=np.asarray(bundle['selected_feature_names']).astype(str)
    observed=np.asarray(external_names)[indices].astype(str)
    if not np.array_equal(observed,expected): raise AssertionError('Conventional selected-feature schema/order mismatch.')
    raw=np.asarray(bundle['estimator'].predict_proba(X[:,indices]),float)
    mapping={str(label):i for i,label in enumerate(bundle['estimator'].classes_)}
    aligned=np.column_stack([raw[:,mapping[label]] for label in bundle['classes']])
    if not np.isfinite(aligned).all() or not np.allclose(aligned.sum(1),1,atol=1e-6): raise AssertionError('Invalid conventional probabilities.')
    return aligned
print('Frozen model reconstruction ready.')


## 5. Generate and persist diagnosis-blind predictions

In [ ]:
blind_checkpoint=TABLES/'05_DIAGNOSIS_BLIND_FROZEN_PREDICTIONS.csv'
expected_groups={(dataset,task,method) for dataset,tasks in EXTERNAL_PROTOCOL.items() for task in tasks for method in METHODS}
blind_rows=[]
if blind_checkpoint.is_file():
    cached=pd.read_csv(blind_checkpoint,dtype={'participant_id':str})
    required={'dataset','task','participant_id','method','predicted_label','decision_threshold'}
    if required.issubset(cached):
        valid=[]
        for (dataset,task,method),q in cached.groupby(['dataset','task','method'],sort=False):
            if (dataset,task,method) not in expected_groups: continue
            classes=TASKS[task]; P=q[[f'probability_{c}' for c in classes]].to_numpy(float)
            expected_ids=set(external[dataset]['blind'].participant_id.astype(str))
            ok=(not q.participant_id.duplicated().any() and set(q.participant_id.astype(str))==expected_ids and
                np.isfinite(P).all() and np.allclose(P.sum(1),1,atol=1e-5))
            if ok: valid.extend(q.to_dict('records'))
        blind_rows=valid
        print('Restored verified diagnosis-blind prediction groups:',
              len({(r['dataset'],r['task'],r['method']) for r in blind_rows}),'of',len(expected_groups))

completed={(r['dataset'],r['task'],r['method']) for r in blind_rows}
conv_index=conv_models.set_index('task'); deep_index=deep_models.set_index('task')
for dataset,tasks in EXTERNAL_PROTOCOL.items():
    data=external[dataset]; blind=data['blind']; rows=blind.feature_row.to_numpy(int)
    for task in tasks:
        classes=TASKS[task]
        need_conv=(dataset,task,METHODS[0]) not in completed
        need_deep=(dataset,task,METHODS[1]) not in completed
        if not (need_conv or need_deep):
            print('restored',dataset,task); continue
        if need_conv:
            conv_bundle=joblib.load(conv_index.loc[task,'model_path'])
            Pconv=conventional_probability(conv_bundle,data['X'][rows],data['feature_names'])
        if need_deep:
            deep_state=load_deep_state(deep_index.loc[task,'model_path'])
            if list(deep_state['classes'])!=classes: raise AssertionError(f'{task}: deep class order mismatch.')
            Pdeep=np.vstack([deep_probability(deep_state,row.array_path) for row in blind.itertuples(index=False)])
        jobs=[]
        if need_conv: jobs.append((METHODS[0],Pconv,None))
        if need_deep: jobs.append((METHODS[1],Pdeep,deep_state.get('threshold')))
        for method,P,threshold in jobs:
            P=np.asarray(P,float); P=np.clip(P,1e-12,1.0); P=P/P.sum(1,keepdims=True)
            if P.shape!=(len(blind),len(classes)) or not np.isfinite(P).all() or not np.allclose(P.sum(1),1,atol=1e-10):
                raise AssertionError(f'Invalid probabilities: {dataset}, {task}, {method}')
            if len(classes)==2:
                cut=.5 if threshold is None else float(threshold); predicted=(P[:,1]>=cut).astype(int)
            else:
                cut=np.nan; predicted=P.argmax(1)
            for i,row in enumerate(blind.itertuples(index=False)):
                item={'dataset':dataset,'task':task,'participant_id':str(row.participant_id),'method':method,
                      'predicted_label':classes[predicted[i]],'decision_threshold':cut}
                item.update({f'probability_{label}':float(P[i,j]) for j,label in enumerate(classes)})
                blind_rows.append(item)
        # Save after every endpoint so an interruption never discards completed inference.
        pd.DataFrame(blind_rows).to_csv(blind_checkpoint,index=False)
        print('checkpointed',dataset,task)

blind_predictions=pd.DataFrame(blind_rows)
observed_groups=set(map(tuple,blind_predictions[['dataset','task','method']].drop_duplicates().to_numpy()))
if observed_groups!=expected_groups: raise AssertionError(f'Incomplete blind prediction groups: {expected_groups-observed_groups}')
if blind_predictions.duplicated(['dataset','task','method','participant_id']).any(): raise AssertionError('Duplicate blind prediction.')
save_table(blind_predictions,'05_DIAGNOSIS_BLIND_FROZEN_PREDICTIONS')
display(blind_predictions.groupby(['dataset','task','method','predicted_label']).size().rename('n').reset_index())
print('Predictions persisted before sealed diagnoses were opened.')


## 6. Open sealed diagnoses and reconcile the five valid external endpoint cohorts

In [ ]:
label_rows=[]
for dataset,spec in DATASETS.items():
    q=pd.read_csv(spec['labels'],dtype={'participant_id':str})
    if not {'participant_id','true_label'}.issubset(q): raise KeyError(f'{dataset} label lock lacks participant_id/true_label.')
    if q.participant_id.duplicated().any(): raise AssertionError(f'{dataset}: duplicate participant IDs in label lock.')
    label_rows.append(q[['participant_id','true_label']].assign(dataset=dataset))
labels=pd.concat(label_rows,ignore_index=True)
evaluation=blind_predictions.merge(labels,on=['dataset','participant_id'],how='left',validate='many_to_one')
if evaluation.true_label.isna().any(): raise AssertionError('At least one prediction lacks a sealed diagnosis.')
evaluation=evaluation.loc[evaluation.apply(lambda row:row.true_label in TASKS[row.task],axis=1)].copy()

cohort_checks=[]
for (dataset,task,method),q in evaluation.groupby(['dataset','task','method'],sort=False):
    classes=TASKS[task]
    cohort_checks.extend([
      {'dataset':dataset,'task':task,'method':method,'check':'all required classes present','pass':set(q.true_label)==set(classes),'detail':str(q.true_label.value_counts().to_dict())},
      {'dataset':dataset,'task':task,'method':method,'check':'one row per participant','pass':not q.participant_id.duplicated().any(),'detail':len(q)},
      {'dataset':dataset,'task':task,'method':method,'check':'valid probabilities','pass':np.isfinite(q[[f'probability_{c}' for c in classes]].to_numpy(float)).all(),'detail':len(q)}])
cohort_checks=pd.DataFrame(cohort_checks)
expected={(d,t) for d,tasks in EXTERNAL_PROTOCOL.items() for t in tasks}
observed=set(map(tuple,evaluation[['dataset','task']].drop_duplicates().to_numpy()))
if observed!=expected or not cohort_checks['pass'].all(): display(cohort_checks); raise AssertionError('External endpoint reconciliation failure.')
save_table(cohort_checks,'06_ENDPOINT_COHORT_RECONCILIATION')
save_table(evaluation,'07_MATCHED_FROZEN_EXTERNAL_PARTICIPANT_PREDICTIONS')
display(evaluation.groupby(['dataset','task','true_label']).participant_id.nunique().rename('participants').reset_index().drop_duplicates())


## 7. Identically computed metrics, participant bootstrap intervals, and paired method differences

In [ ]:
def ece_binary(y,p,bins=10):
    edges=np.linspace(0,1,bins+1); value=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        use=(p>=lo)&(p<(hi if hi<1 else hi+1e-12))
        if use.any(): value+=use.mean()*abs(y[use].mean()-p[use].mean())
    return float(value)

def evaluate(q,task):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}
    y=q.true_label.map(c2i).to_numpy(int); yp=q.predicted_label.map(c2i).to_numpy(int)
    P=q[[f'probability_{c}' for c in classes]].to_numpy(float)
    # Exact normalization prevents log-loss warning floods from float32 rounding.
    P=np.clip(P,1e-12,1.0); P=P/P.sum(1,keepdims=True)
    out={'n':len(q),'accuracy':accuracy_score(y,yp),'balanced_accuracy':balanced_accuracy_score(y,yp),
         'macro_f1':f1_score(y,yp,average='macro',zero_division=0),'mcc':matthews_corrcoef(y,yp),
         'log_loss':log_loss(y,P,labels=np.arange(len(classes)))}
    onehot=np.eye(len(classes))[y]; out['multiclass_brier']=float(np.mean(np.sum((P-onehot)**2,axis=1)))
    if len(classes)==2:
        tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
        out.update({'sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp),'roc_auc':roc_auc_score(y,P[:,1]),
                    'binary_brier':brier_score_loss(y,P[:,1]),'ece_10bin':ece_binary(y,P[:,1])})
    else:
        out['roc_auc']=roc_auc_score(y,P,multi_class='ovr',average='macro',labels=np.arange(len(classes)))
    return out

def safe_name(value): return str(value).lower().replace(' ','_').replace('/','_')
def checkpointed_bootstrap(q,task,checkpoint,seed):
    q=q.reset_index(drop=True); classes=TASKS[task]; strata={c:np.flatnonzero(q.true_label.to_numpy()==c) for c in classes}
    if checkpoint.is_file():
        boot=pd.read_csv(checkpoint)
        if len(boot)>BOOTSTRAP_REPS: boot=boot.iloc[:BOOTSTRAP_REPS].copy()
    else: boot=pd.DataFrame()
    start=len(boot); rows=[]
    for repetition in range(start,BOOTSTRAP_REPS):
        rng=np.random.default_rng(int(seed)+repetition)
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes])
        rows.append({'bootstrap_repetition':repetition,**evaluate(q.iloc[ix],task)})
        if len(rows)>=100 or repetition==BOOTSTRAP_REPS-1:
            boot=pd.concat([boot,pd.DataFrame(rows)],ignore_index=True); rows=[]
            boot.to_csv(checkpoint,index=False)
    return boot

metric_rows=[]; interval_rows=[]
group_order=sorted(evaluation[['dataset','task','method']].drop_duplicates().itertuples(index=False,name=None))
for group_number,(dataset,task,method) in enumerate(group_order,1):
    q=evaluation.loc[(evaluation.dataset==dataset)&(evaluation.task==task)&(evaluation.method==method)].reset_index(drop=True)
    observed_metrics=evaluate(q,task); metric_rows.append({'dataset':dataset,'task':task,'method':method,**observed_metrics})
    checkpoint=CHECKPOINTS/f"bootstrap__{safe_name(dataset)}__{task}__{safe_name(method)}.csv"
    boot=checkpointed_bootstrap(q,task,checkpoint,SEED+100000*group_number)
    for metric,estimate in observed_metrics.items():
        if metric!='n':
            values=boot[metric].dropna(); interval_rows.append({'dataset':dataset,'task':task,'method':method,'metric':metric,
              'estimate':estimate,'ci95_low':values.quantile(.025),'ci95_high':values.quantile(.975)})
    print(f'bootstrap complete {group_number}/{len(group_order)}:',dataset,task,method)
metrics=pd.DataFrame(metric_rows); intervals=pd.DataFrame(interval_rows)

def holm_adjust(values):
    p=np.asarray(values,float); order=np.argsort(p); adjusted=np.empty(len(p)); running=0.0
    for rank,index in enumerate(order): running=max(running,(len(p)-rank)*p[index]); adjusted[index]=min(1.0,running)
    return adjusted

paired=[]
pair_order=sorted({(d,t) for d,tasks in EXTERNAL_PROTOCOL.items() for t in tasks})
for pair_number,(dataset,task) in enumerate(pair_order,1):
    a=evaluation.loc[(evaluation.dataset==dataset)&(evaluation.task==task)&(evaluation.method==METHODS[0])].set_index('participant_id').sort_index()
    b=evaluation.loc[(evaluation.dataset==dataset)&(evaluation.task==task)&(evaluation.method==METHODS[1])].set_index('participant_id').sort_index()
    if not a.index.equals(b.index) or not a.true_label.equals(b.true_label): raise AssertionError(f'Unmatched methods: {dataset}, {task}')
    classes=TASKS[task]; strata={c:np.flatnonzero(a.true_label.to_numpy()==c) for c in classes}
    checkpoint=CHECKPOINTS/f'paired__{safe_name(dataset)}__{task}.csv'
    if checkpoint.is_file(): delta_frame=pd.read_csv(checkpoint).iloc[:BOOTSTRAP_REPS].copy()
    else: delta_frame=pd.DataFrame(columns=['bootstrap_repetition','delta_ba'])
    rows=[]
    for repetition in range(len(delta_frame),BOOTSTRAP_REPS):
        rng=np.random.default_rng(SEED+900000+100000*pair_number+repetition)
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes])
        delta=evaluate(b.iloc[ix],task)['balanced_accuracy']-evaluate(a.iloc[ix],task)['balanced_accuracy']
        rows.append({'bootstrap_repetition':repetition,'delta_ba':delta})
        if len(rows)>=100 or repetition==BOOTSTRAP_REPS-1:
            delta_frame=pd.concat([delta_frame,pd.DataFrame(rows)],ignore_index=True); rows=[]
            delta_frame.to_csv(checkpoint,index=False)
    delta=delta_frame.delta_ba.to_numpy(float)
    observed=evaluate(b,task)['balanced_accuracy']-evaluate(a,task)['balanced_accuracy']
    p=min(1.0,2*min((np.sum(delta<=0)+1)/(len(delta)+1),(np.sum(delta>=0)+1)/(len(delta)+1)))
    paired.append({'dataset':dataset,'task':task,'selected_minus_conventional_ba':observed,
                   'ci95_low':np.quantile(delta,.025),'ci95_high':np.quantile(delta,.975),'unadjusted_p_value':p})
    print(f'paired bootstrap complete {pair_number}/{len(pair_order)}:',dataset,task)
paired=pd.DataFrame(paired); paired['holm_adjusted_p_value']=holm_adjust(paired.unadjusted_p_value)
save_table(metrics,'08_EXTERNAL_METRICS'); save_table(intervals,'09_PARTICIPANT_BOOTSTRAP_INTERVALS')
save_table(paired,'10_PAIRED_EXTERNAL_METHOD_DIFFERENCES')
display(metrics.round(4)); display(paired.round(4))


## 8. Feature shift and probability shift diagnostics

In [ ]:
# Shift is diagnostic, not a causal explanation. It does not modify any prediction.
development_feature_path=Path(conv_run.get('input_feature_path','')) if conv_run.get('input_feature_path') else None
if development_feature_path is None or not development_feature_path.is_file():
    candidate=LEGACY_OUTPUTS_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'/'features'/'02_LOCKED_RECORDING_LEVEL_NONSPIKING_FEATURE_MATRIX.npz'
    development_feature_path=candidate if candidate.is_file() else None

shift_rows=[]
if development_feature_path is not None:
    with np.load(development_feature_path,allow_pickle=False) as z:
        Xdev=np.asarray(z['X'],float); dev_names=z['feature_names'].astype(str)
    for dataset,data in external.items():
        if not np.array_equal(dev_names,data['feature_names']): raise AssertionError(f'{dataset}: full feature schema differs from development.')
        Xext=np.asarray(data['X'],float)
        mean_dev=Xdev.mean(0); sd_dev=Xdev.std(0,ddof=1); mean_ext=Xext.mean(0); sd_ext=Xext.std(0,ddof=1)
        pooled=np.sqrt((sd_dev**2+sd_ext**2)/2); smd=np.divide(mean_ext-mean_dev,pooled,out=np.zeros_like(mean_dev),where=pooled>1e-12)
        for j,name in enumerate(dev_names): shift_rows.append({'dataset':dataset,'feature_name':name,'smd_external_minus_development':smd[j],'absolute_smd':abs(smd[j])})
feature_shift=pd.DataFrame(shift_rows)
if len(feature_shift): save_table(feature_shift,'11_FEATURE_SHIFT_STANDARDIZED_MEAN_DIFFERENCES')

probability_shift=[]
for (dataset,task,method),q in evaluation.groupby(['dataset','task','method']):
    classes=TASKS[task]
    for label in classes:
        values=q.loc[q.true_label.eq(label),f'probability_{label}'].to_numpy(float)
        probability_shift.append({'dataset':dataset,'task':task,'method':method,'true_class':label,'n':len(values),
                                  'mean_true_class_probability':values.mean(),'sd_true_class_probability':values.std(ddof=1) if len(values)>1 else np.nan,
                                  'median_true_class_probability':np.median(values)})
probability_shift=pd.DataFrame(probability_shift); save_table(probability_shift,'12_EXTERNAL_PROBABILITY_DISTRIBUTION_SUMMARY')
display(feature_shift.groupby('dataset').absolute_smd.agg(['median','mean','max']).round(3) if len(feature_shift) else pd.DataFrame({'note':['Development feature matrix not resolved; feature-shift table not generated.']}))
display(probability_shift.round(4))


## 9. Publication-ready figures

In [ ]:
# Figure 1: external balanced accuracy with endpoint-specific chance markers.
ba=intervals.loc[intervals.metric.eq('balanced_accuracy')].copy()
ba['display']=ba.apply(lambda r:f"{r['dataset']} · {TASK_LABELS[r['task']]} · {'Conventional' if r['method']==METHODS[0] else 'Selected deep'}",axis=1)
ba=ba.sort_values(['dataset','task','method']).reset_index(drop=True); y=np.arange(len(ba))
fig,ax=plt.subplots(figsize=(10,7)); values=ba.estimate.to_numpy(float)
err=np.vstack([values-ba.ci95_low.to_numpy(float),ba.ci95_high.to_numpy(float)-values])
colors=['#4C78A8' if m==METHODS[0] else '#F28E2B' for m in ba.method]
for i in range(len(ba)):
    ax.errorbar(values[i],y[i],xerr=np.asarray([[err[0,i]],[err[1,i]]]),fmt='o',color=colors[i],capsize=3)
    ax.scatter(CHANCE_BA[ba.loc[i,'task']],y[i],marker='|',s=180,color='#B22222')
ax.set_yticks(y); ax.set_yticklabels(ba.display); ax.set_xlim(.25,1.02)
ax.set_xlabel('Balanced accuracy (95% participant-bootstrap CI)'); ax.set_title('Frozen external re-evaluation')
ax.grid(axis='y',visible=False); save_figure(fig,'Figure_1_External_Balanced_Accuracy'); plt.show()

# Figure 2: matched method difference; values above zero favor selected deep models.
q=paired.sort_values(['dataset','task']).reset_index(drop=True); y=np.arange(len(q)); value=q.selected_minus_conventional_ba.to_numpy(float)
err=np.vstack([value-q.ci95_low.to_numpy(float),q.ci95_high.to_numpy(float)-value])
fig,ax=plt.subplots(figsize=(9,5)); ax.errorbar(value,y,xerr=err,fmt='o',capsize=4,color='#1F4E79'); ax.axvline(0,color='0.3',ls='--')
ax.set_yticks(y); ax.set_yticklabels([f"{d} · {TASK_LABELS[t]}" for d,t in zip(q.dataset,q.task)])
ax.set_xlabel('Selected deep minus conventional balanced accuracy'); ax.set_title('Matched external method differences')
ax.grid(axis='y',visible=False); save_figure(fig,'Figure_2_Paired_External_Method_Differences'); plt.show()

# Figure 3: binary ROC curves, separated by method style.
fig,ax=plt.subplots(figsize=(8,6)); palette={'AHEPA':'#0072B2','Dryad':'#D55E00'}; styles={METHODS[0]:'-',METHODS[1]:'--'}
for (dataset,task,method),frame in evaluation.groupby(['dataset','task','method']):
    classes=TASKS[task]
    if len(classes)!=2: continue
    ytrue=(frame.true_label.to_numpy()==classes[1]).astype(int); prob=frame[f'probability_{classes[1]}'].to_numpy(float)
    fpr,tpr,_=roc_curve(ytrue,prob); auc=roc_auc_score(ytrue,prob)
    ax.plot(fpr,tpr,color=palette[dataset],ls=styles[method],lw=1.8,label=f"{dataset} · {TASK_LABELS[task]} · {'Conv.' if method==METHODS[0] else 'Deep'} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],color='0.5',ls=':'); ax.set(xlabel='False-positive rate',ylabel='True-positive rate',title='Frozen external binary discrimination')
ax.legend(frameon=False,fontsize=7); save_figure(fig,'Figure_3_External_ROC_Curves'); plt.show()

# Figure 4: probability histograms for the five selected-deep external endpoints.
deep_eval=evaluation.loc[evaluation.method.eq(METHODS[1])].copy(); panels=sorted(expected); fig,axes=plt.subplots(3,2,figsize=(11,11)); axes=axes.ravel()
for ax,(dataset,task) in zip(axes,panels):
    frame=deep_eval.query('dataset==@dataset and task==@task'); classes=TASKS[task]
    if len(classes)==2:
        sns.histplot(data=frame,x=f'probability_{classes[1]}',hue='true_label',hue_order=classes,bins=12,element='step',common_norm=False,alpha=.2,ax=ax)
        ax.set_xlabel(f'Frozen P({classes[1]})')
    else:
        confidence=frame[[f'probability_{c}' for c in classes]].max(axis=1)
        ax.hist(confidence,bins=12,color='#59A14F',alpha=.75); ax.set_xlabel('Maximum predicted probability')
    ax.set_title(f'{dataset} · {TASK_LABELS[task]}'); ax.set_ylabel('Count')
for ax in axes[len(panels):]: ax.axis('off')
save_figure(fig,'Figure_4_External_Probability_Distributions'); plt.show()

# Figure 5: feature-shift distribution if the development feature matrix is available.
if len(feature_shift):
    fig,ax=plt.subplots(figsize=(8,4.8)); sns.histplot(data=feature_shift,x='absolute_smd',hue='dataset',element='step',bins=30,common_norm=False,ax=ax)
    ax.axvline(.5,color='#B22222',ls='--',label='|SMD| = 0.5'); ax.set_xlabel('Absolute standardized mean difference'); ax.set_title('External feature shift relative to CAUEEG development')
    save_figure(fig,'Figure_5_External_Feature_Shift'); plt.show()


## 10. Final integrity, provenance, and claim lock

In [ ]:
checks=[]
for dataset,tasks in EXTERNAL_PROTOCOL.items():
    for task in tasks:
        classes=TASKS[task]
        a=evaluation.loc[(evaluation.dataset==dataset)&(evaluation.task==task)&(evaluation.method==METHODS[0])].sort_values('participant_id')
        b=evaluation.loc[(evaluation.dataset==dataset)&(evaluation.task==task)&(evaluation.method==METHODS[1])].sort_values('participant_id')
        checks.extend([
          {'scope':f'{dataset}:{task}','check':'matched participants and labels across methods','pass':a.participant_id.reset_index(drop=True).equals(b.participant_id.reset_index(drop=True)) and a.true_label.reset_index(drop=True).equals(b.true_label.reset_index(drop=True))},
          {'scope':f'{dataset}:{task}','check':'one prediction per participant per method','pass':len(a)>0 and not a.participant_id.duplicated().any() and not b.participant_id.duplicated().any()},
          {'scope':f'{dataset}:{task}','check':'correct mathematical chance reference','pass':np.isclose(CHANCE_BA[task],.5 if len(classes)==2 else 1/3)}])
checks.extend([
 {'scope':'all','check':'five compatible dataset-endpoint combinations evaluated','pass':set(map(tuple,evaluation[['dataset','task']].drop_duplicates().to_numpy()))==expected},
 {'scope':'all','check':'both frozen model families evaluated','pass':set(evaluation.method)==set(METHODS)},
 {'scope':'all','check':'all frozen model hashes authenticated','pass':model_checks.authenticated.all()},
 {'scope':'all','check':'diagnosis-blind predictions persisted before labels opened','pass':(TABLES/'05_DIAGNOSIS_BLIND_FROZEN_PREDICTIONS.csv').is_file()},
 {'scope':'all','check':'no model refitting on external data','pass':True},
 {'scope':'all','check':'no external feature selection','pass':True},
 {'scope':'all','check':'no external threshold optimization','pass':True},
 {'scope':'all','check':'no external recalibration or adaptation','pass':True},
 {'scope':'all','check':'historically untouched external claim prohibited','pass':not chronology.historically_untouched_for_P1_E16_7E.any()},
 {'scope':'all','check':'participant-stratified bootstrap complete','pass':len(intervals)>0 and BOOTSTRAP_REPS==5000},
 {'scope':'all','check':'no performance target used as gate','pass':True}])
checks=pd.DataFrame(checks); save_table(checks,'13_FINAL_INTEGRITY_CHECKS'); display(checks)
if not checks['pass'].all(): raise AssertionError('P1-E16.7E integrity failure.')

artifacts=[]
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file() and path.name!='14_OUTPUT_MANIFEST.csv':
        artifacts.append({'relative_path':str(path.relative_to(OUTPUT_ROOT)),'bytes':path.stat().st_size,'sha256':sha256_file(path)})
artifact_manifest=pd.DataFrame(artifacts); save_table(artifact_manifest,'14_OUTPUT_MANIFEST')
run={'experiment':'P1-E16.7E frozen external re-evaluation, transportability, shift, and chronology audit',
     'status':'PASSED','completed_utc':datetime.now(timezone.utc).isoformat(),'endpoints':TASKS,
     'external_protocol':EXTERNAL_PROTOCOL,'methods':METHODS,
     'evidence_label':'locked external re-evaluation under final P1-E16.7A/P1-E16.7D contracts',
     'historically_untouched_claim':False,'external_refit':False,'external_threshold_optimization':False,
     'external_recalibration':False,'external_adaptation':False,'performance_target_used':False,
     'bootstrap_repetitions':BOOTSTRAP_REPS,'integrity_passed':True,
     'metrics':metrics.to_dict('records'),'paired_differences':paired.to_dict('records'),
     'environment':{'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'device':str(DEVICE)}}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2,default=json_default),encoding='utf-8')
print('P1-E16.7E PASSED — FROZEN EXTERNAL RE-EVALUATION AND TRANSPORTABILITY AUDIT LOCKED')
